In [ ]:
import lightgbm as lgb
import sklearn

print("LightGBM:", lgb.__version__)
print("Scikit-learn:", sklearn.__version__)

In [ ]:
%load_ext autoreload
%autoreload 2

from gridcast.models import SeasonalNaive, Blend, LightGBMModel

Preapre dataset with features

In [ ]:
import copy
from gridcast.config import load_config
from gridcast.data import load_raw, make_regular
from gridcast.features import build_features
from gridcast.models import LightGBMModel
import pandas as pd
cfg = load_config("../configs/config.yaml")

cfg_demand = copy.deepcopy(cfg)
cfg_demand["features"]["use_weather"] = False      # demand-only baseline

raw = load_raw("../" + cfg["data"]["raw_path"])
df = make_regular(raw, cfg["data"]["freq"], cfg["data"]["max_gap_steps"])
X, y = build_features(df, cfg_demand)



FEATURE_GROUPS = {
    "calendar": ["interval_of_day", "day_of_week", "is_weekend"],
    "holidays": ["is_holiday", "days_to_holiday", "days_since_holiday"],
    "demand_lags": ["demand_met_mw_lag96", "demand_met_mw_lag192", "demand_met_mw_lag672"],
    "rolling": ["demand_roll96_mean", "demand_roll96_std",
                "demand_roll672_mean", "demand_roll672_std"],
    "gen_lags": ["solar_mw_lag96", "solar_mw_lag672", "wind_mw_lag96", "wind_mw_lag672"],
    "weather": ["weighted_temperature_2m", "weighted_relative_humidity_2m",
                "weighted_apparent_temperature", "weighted_dew_point_2m",
                "cdd", "app_cdd", "temp_day_max", "temp_day_mean", "temp_max_3d_mean"],
}


def select_groups(X: pd.DataFrame, groups: list[str]) -> pd.DataFrame:
    cols = [c for g in groups for c in FEATURE_GROUPS[g]]
    missing = [c for c in cols if c not in X.columns]
    if missing:
        raise KeyError(f"Columns not in X (is use_weather false, or was a feature renamed?): {missing}")
    return X[cols].copy()



ablation = {
    "calendar only":      ["calendar"],
    "+ demand lags":      ["calendar", "demand_lags"],
    "+ rolling":          ["calendar", "demand_lags", "rolling"],
    "+ holidays":         ["calendar", "demand_lags", "rolling", "holidays"],
    "+ weather":          ["calendar", "demand_lags", "rolling", "holidays", "weather"],
}

In [ ]:
from gridcast.backtest import make_folds
from gridcast.models import LightGBMModel

# Create the same chronological folds used by our backtest
bt = cfg["backtest"]

folds = make_folds(
    X.index,
    n_folds=bt["n_folds"],
    test_days=bt["test_days"],
    min_train_days=bt["min_train_days"],
)

# Select the first fold
train_idx, test_idx, test_start = folds[0]

X_train = X.loc[train_idx]
y_train = y.loc[train_idx]

# Fit the model; its final 28 training days are used for validation
model = LightGBMModel(
    random_state=cfg["seed"],
    validation_days=28,
)

model.fit(X_train, y_train)

# Predict the held-out test fold
pred = model.predict(X.loc[test_idx])

print("Test start:", test_start)
print("Training rows:", len(X_train))
print("Test rows:", len(test_idx))
print("Best iteration:", model.best_iteration_)
print("Predictions:", len(pred))
print("Prediction range:", round(pred.min(), 2), "to", round(pred.max(), 2))
print("Actual range:", round(y.loc[test_idx].min(), 2), "to", round(y.loc[test_idx].max(), 2))

In [ ]:
import pandas as pd
from gridcast.metrics import evaluate

pred_series = pd.Series(pred, index=test_idx)

peak = (
    bt["peak_start_hour"],
    bt["peak_end_hour"],
)

lgb_metrics = evaluate(
    y.loc[test_idx],
    pred_series,
    y_train,
    peak,
)

naive_metrics = evaluate(
    y.loc[test_idx],
    X.loc[test_idx, "demand_met_mw_lag96"],
    y_train,
    peak,
)

print("LightGBM:", lgb_metrics)
print("Naive 1-day:", naive_metrics)

In [ ]:
from gridcast.backtest import run_backtest
from gridcast.models import LightGBMModel

lgb_model = LightGBMModel(
    random_state=cfg["seed"],
    validation_days=28,
)

lgb_fold_metrics, lgb_preds = run_backtest(
    lgb_model,
    X,
    y,
    cfg,
)

print("Per-fold LightGBM results:")
print(lgb_fold_metrics.to_string(index=False))

print("\nMean metrics across folds:")
print(
    lgb_fold_metrics[
        ["mae", "rmse", "mape", "mase", "peak_mape"]
    ].mean().round(3)
)

print("\nTotal predictions:", len(lgb_preds))

In [ ]:
# Inspect fold 6 predictions (May 17, 2026)
fold6 = lgb_preds[lgb_preds["fold"] == 6].copy()

fold6["error"] = fold6["pred"] - fold6["y"]
fold6["abs_error"] = fold6["error"].abs()
fold6["ape"] = (
    fold6["abs_error"] / fold6["y"].abs()
) * 100

print("Fold 6 summary:")
print(fold6[["y", "pred", "error", "ape"]].describe().round(2))

print("\nWorst 10 timestamps:")
print(
    fold6.nlargest(10, "ape")[
        ["y", "pred", "error", "ape"]
    ].round(2)
)

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# Prepare data
fold6 = lgb_preds[lgb_preds["fold"] == 6].copy()

fold6["error"] = fold6["pred"] - fold6["y"]
fold6["ape"] = (
    fold6["error"].abs() / fold6["y"].abs()
) * 100

# --------------------------------
# Plot 1: Actual vs predicted demand
# --------------------------------
plt.figure(figsize=(15, 5))

plt.plot(fold6.index, fold6["y"], label="Actual")
plt.plot(fold6.index, fold6["pred"], label="LightGBM prediction")

plt.title("Fold 6: Actual vs Predicted Demand")
plt.xlabel("Date")
plt.ylabel("Demand (MW)")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


# --------------------------------
# Plot 2: Prediction error over time
# --------------------------------
plt.figure(figsize=(15, 4))

plt.plot(fold6.index, fold6["error"], label="Prediction error")
plt.axhline(0, linestyle="--")

plt.title("Fold 6: Prediction Error (Predicted - Actual)")
plt.xlabel("Date")
plt.ylabel("Error (MW)")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


# --------------------------------
# Plot 3: Daily MAPE
# --------------------------------
daily_mape = fold6["ape"].resample("D").mean()

plt.figure(figsize=(13, 4))

plt.plot(daily_mape.index, daily_mape.values, marker="o")

plt.title("Fold 6: Daily MAPE")
plt.xlabel("Date")
plt.ylabel("MAPE (%)")
plt.grid(True)
plt.tight_layout()
plt.show()


# --------------------------------
# Plot 4: Average demand by hour
# --------------------------------
hourly = fold6.groupby(fold6.index.hour)[["y", "pred"]].mean()

plt.figure(figsize=(11, 5))

plt.plot(hourly.index, hourly["y"], marker="o", label="Actual")
plt.plot(hourly.index, hourly["pred"], marker="o", label="Predicted")

plt.title("Fold 6: Average Demand by Hour of Day")
plt.xlabel("Hour")
plt.ylabel("Average Demand (MW)")
plt.xticks(range(0, 24))
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# Hourly average signed error
hourly_error = fold6["error"].groupby(fold6.index.hour).mean()

plt.figure(figsize=(12, 4))
plt.bar(hourly_error.index, hourly_error.values)
plt.axhline(0, linestyle="--")
plt.title("Fold 6: Average Prediction Bias by Hour")
plt.xlabel("Hour of Day")
plt.ylabel("Predicted - Actual (MW)")
plt.xticks(range(24))
plt.grid(axis="y")
plt.tight_layout()
plt.show()


# Daily peak demand: actual vs predicted
daily_peak = fold6[["y", "pred"]].resample("D").max()

plt.figure(figsize=(14, 5))
plt.plot(daily_peak.index, daily_peak["y"], marker="o", label="Actual daily peak")
plt.plot(daily_peak.index, daily_peak["pred"], marker="o", label="Predicted daily peak")

plt.title("Fold 6: Daily Peak Demand")
plt.xlabel("Date")
plt.ylabel("Peak Demand (MW)")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


# Daily average signed error
daily_bias = fold6["error"].resample("D").mean()

plt.figure(figsize=(14, 4))
plt.bar(daily_bias.index, daily_bias.values)
plt.axhline(0, linestyle="--")
plt.title("Fold 6: Daily Average Prediction Bias")
plt.xlabel("Date")
plt.ylabel("Predicted - Actual (MW)")
plt.grid(axis="y")
plt.tight_layout()
plt.show()

Add Temperature and other weather param as a feature in lightBGM

In [ ]:
from gridcast.config import load_config
from gridcast.weather import build_weighted_weather, build_weather_history, load_weather, weather_table_path

cfg = load_config("../configs/config.yaml")        # validates; adds cfg["root"]

w = build_weighted_weather("2025-06-10", cfg)      # one day, 8 cities
print(w.shape)
print(w.iloc[[0, 32, 56, 80]].round(1))            # 00:00, 08:00, 14:00, 20:00         # 8 API calls, then cached
print(w.shape)                                         # (96, 4)
print(w.iloc[[0, 32, 56, 80]].round(1))                # 00:00, 08:00, 14:00, 20:00

days = df.index.normalize().unique()
weather, failures = build_weather_history(days, cfg)   # ~5,800 calls, ~100 min at 1 call/s
print(failures[:5])

# Coverage check before modeling
weather = load_weather(cfg["weather"]["features_path"])
print("demand rows with weather:", df.index.isin(weather.index).mean().round(3))

X, y = build_features(df, cfg, weather=weather)
print(X.filter(regex="weighted|cdd|temp_").isna().mean().round(3))